# 🥈 Silver Layer — Data Cleaning & Conforming
**LushProtein Analytics | Medallion Architecture**

Reads from `medallion/bronze/`, applies cleaning & type casting, writes to `medallion/silver/`.

| Bronze Table | Silver Table | Key Cleaning |
|---|---|---|
| `bronze_orders` | `silver_orders` | Parse dates/numerics, deduplicate, normalise categoricals |
| `bronze_products` | `silver_products` | Normalise SKU, strip whitespace, handle nulls |
| `bronze_discounts` | `silver_discounts` | Parse dates, normalise discount types |
| `bronze_sessions` | `silver_sessions` | Parse dates/numerics, normalise referrer |
| `bronze_rc_orders` | `silver_rc_orders` | Parse dates/numerics, normalise status |
| `bronze_rc_items_checkout` | `silver_rc_items_checkout` | Parse numerics, validate SKU references |
| `bronze_rc_items_recurring` | `silver_rc_items_recurring` | Parse numerics, validate SKU references |
| `bronze_rc_reactivated` | `silver_rc_reactivated` | Parse dates, deduplicate |
| `bronze_rc_churned` | `silver_rc_churned` | Parse dates, deduplicate |

## 0. Setup & Paths

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_colwidth', 80)

BASE       = os.path.dirname(os.path.abspath('__file__'))
BRONZE_DIR = os.path.join(BASE, 'medallion', 'bronze')
SILVER_DIR = os.path.join(BASE, 'medallion', 'silver')
os.makedirs(SILVER_DIR, exist_ok=True)

print(f"Bronze : {BRONZE_DIR}")
print(f"Silver : {SILVER_DIR}")

### Helper utilities

In [ ]:
def cleaning_report(df_before, df_after, table_name):
    """Print a before/after summary after cleaning a table."""
    rows_dropped = df_before.shape[0] - df_after.shape[0]
    dup_before   = df_before.duplicated().sum()
    null_before  = df_before.isnull().mean().mean()
    null_after   = df_after.isnull().mean().mean()
    print(f"\n{'─'*55}")
    print(f"  {table_name}")
    print(f"{'─'*55}")
    print(f"  Rows        : {df_before.shape[0]:>7,}  →  {df_after.shape[0]:>7,}  (dropped {rows_dropped:,})")
    print(f"  Duplicates  : {dup_before:>7,}  →  0")
    print(f"  Avg null%   : {null_before:>7.1%}  →  {null_after:>7.1%}")
    print(f"  Cols        : {df_before.shape[1]:>7}  →  {df_after.shape[1]:>7}")


def normalise_str_col(series):
    """Strip whitespace and title-case a string column."""
    return series.str.strip().str.title()


def safe_to_numeric(series):
    """Coerce to numeric; non-parseable values become NaN."""
    return pd.to_numeric(series, errors='coerce')


def safe_to_datetime(series, utc=True):
    """Parse datetime strings; non-parseable values become NaT."""
    dt = pd.to_datetime(series, errors='coerce', utc=utc)
    return dt


print("✅ Helpers loaded")

---
## 1. Orders (`bronze_orders` → `silver_orders`)

In [ ]:
df = pd.read_parquet(os.path.join(BRONZE_DIR, 'bronze_orders.parquet'))
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

In [ ]:
# ── 1.1  Drop fully empty rows ────────────────────────────────────────────────
df = df.dropna(how='all')

# ── 1.2  Deduplicate on order ID ─────────────────────────────────────────────
# Keep first occurrence; adjust 'last' if you prefer most-recent source file
df = df.drop_duplicates(subset=['ID'], keep='first')

# ── 1.3  Cast date columns ────────────────────────────────────────────────────
date_cols = ['Processed At', 'Created At', 'Updated At', 'Cancelled At']
for col in date_cols:
    if col in df.columns:
        df[col] = safe_to_datetime(df[col])

# ── 1.4  Cast numeric columns ─────────────────────────────────────────────────
numeric_cols = [
    'Total', 'Subtotal', 'Shipping', 'Taxes', 'Refunded Amount',
    'Outstanding Balance', 'Discount Amount', 'Line: Quantity',
    'Line: Price', 'Line: Compare At Price'
]
for col in numeric_cols:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

# ── 1.5  Normalise categoricals ───────────────────────────────────────────────
cat_cols = ['Financial Status', 'Fulfillment Status', 'Source', 'Line: Type', 'Currency']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].str.strip().str.lower()

# ── 1.6  Drop rows with no order ID or negative totals ────────────────────────
df = df[df['ID'].notna()]
if 'Total' in df.columns:
    invalid_total = df['Total'] < 0
    print(f"Rows with negative Total: {invalid_total.sum():,} → flagged as 'refund_only'")
    df.loc[invalid_total, 'order_flag'] = 'refund_only'

# ── 1.7  Add derived time columns ─────────────────────────────────────────────
if 'Processed At' in df.columns:
    df['order_year']  = df['Processed At'].dt.year
    df['order_month'] = df['Processed At'].dt.month
    df['order_date']  = df['Processed At'].dt.date

silver_orders = df
cleaning_report(df_raw, silver_orders, 'silver_orders')

In [ ]:
# ── Quick dtype check ─────────────────────────────────────────────────────────
print(silver_orders.dtypes)
display(silver_orders.head(3))

silver_orders.to_parquet(os.path.join(SILVER_DIR, 'silver_orders.parquet'), index=False)
print("\n✅ Saved: silver_orders.parquet")

---
## 2. Products (`bronze_products` → `silver_products`)

In [ ]:
df = pd.read_parquet(os.path.join(BRONZE_DIR, 'bronze_products.parquet'))
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

In [ ]:
# ── 2.1  Drop fully empty rows ────────────────────────────────────────────────
df = df.dropna(how='all')

# ── 2.2  Normalise SKU — strip whitespace, uppercase ──────────────────────────
if 'Variant SKU' in df.columns:
    df['Variant SKU'] = df['Variant SKU'].str.strip().str.upper()

# ── 2.3  Deduplicate on SKU (keep first) ─────────────────────────────────────
if 'Variant SKU' in df.columns:
    df = df.drop_duplicates(subset=['Variant SKU'], keep='first')

# ── 2.4  Cast numeric pricing columns ────────────────────────────────────────
price_cols = ['Variant Price', 'Variant Compare At Price', 'Variant Grams', 'Variant Inventory Qty']
for col in price_cols:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

# ── 2.5  Normalise text fields ────────────────────────────────────────────────
for col in ['Title', 'Product Type', 'Vendor', 'Status']:
    if col in df.columns:
        df[col] = df[col].str.strip()

if 'Status' in df.columns:
    df['Status'] = df['Status'].str.lower()

silver_products = df
cleaning_report(df_raw, silver_products, 'silver_products')

In [ ]:
display(silver_products.head(3))
silver_products.to_parquet(os.path.join(SILVER_DIR, 'silver_products.parquet'), index=False)
print("✅ Saved: silver_products.parquet")

---
## 3. Discounts (`bronze_discounts` → `silver_discounts`)

In [ ]:
df = pd.read_parquet(os.path.join(BRONZE_DIR, 'bronze_discounts.parquet'))
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

In [ ]:
# ── 3.1  Deduplicate on discount code Name ────────────────────────────────────
if 'Name' in df.columns:
    df['Name'] = df['Name'].str.strip().str.upper()
    df = df.drop_duplicates(subset=['Name'], keep='first')

# ── 3.2  Parse date columns ───────────────────────────────────────────────────
for col in ['Starts At', 'Ends At', 'Created At']:
    if col in df.columns:
        df[col] = safe_to_datetime(df[col])

# ── 3.3  Cast numeric value columns ──────────────────────────────────────────
for col in ['Value', 'Minimum Order Amount', 'Times Used']:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

# ── 3.4  Normalise type/status ────────────────────────────────────────────────
for col in ['Type', 'Status', 'Value Type']:
    if col in df.columns:
        df[col] = df[col].str.strip().str.lower()

silver_discounts = df
cleaning_report(df_raw, silver_discounts, 'silver_discounts')

silver_discounts.to_parquet(os.path.join(SILVER_DIR, 'silver_discounts.parquet'), index=False)
print("✅ Saved: silver_discounts.parquet")

---
## 4. Sessions / Traffic (`bronze_sessions` → `silver_sessions`)

In [ ]:
df = pd.read_parquet(os.path.join(BRONZE_DIR, 'bronze_sessions.parquet'))
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

In [ ]:
# ── 4.1  Parse date / period column ──────────────────────────────────────────
for col in ['Day', 'Date', 'Period']:
    if col in df.columns:
        df[col] = safe_to_datetime(df[col], utc=False)

# ── 4.2  Cast numeric metric columns ─────────────────────────────────────────
metric_cols = ['Sessions', 'Visitors', 'Bounce Rate', 'Conversion Rate',
               'Orders', 'Revenue', 'Average Order Value']
for col in metric_cols:
    if col in df.columns:
        # Strip % signs if present, then cast
        df[col] = df[col].astype(str).str.replace('%', '', regex=False)
        df[col] = safe_to_numeric(df[col])

# ── 4.3  Normalise referrer/source strings ────────────────────────────────────
for col in ['Referrer Source', 'Source', 'Referrer Name', 'Channel']:
    if col in df.columns:
        df[col] = df[col].str.strip().str.lower()

# ── 4.4  Drop fully empty rows ────────────────────────────────────────────────
df = df.dropna(how='all')

silver_sessions = df
cleaning_report(df_raw, silver_sessions, 'silver_sessions')

silver_sessions.to_parquet(os.path.join(SILVER_DIR, 'silver_sessions.parquet'), index=False)
print("✅ Saved: silver_sessions.parquet")

---
## 5. Recharge Orders (`bronze_rc_orders` → `silver_rc_orders`)

In [ ]:
df = pd.read_parquet(os.path.join(BRONZE_DIR, 'bronze_rc_orders.parquet'))
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
display(df.head(3))

In [ ]:
# ── 5.1  Deduplicate on recharge_order_id ─────────────────────────────────────
if 'recharge_order_id' in df.columns:
    df = df.drop_duplicates(subset=['recharge_order_id'], keep='first')

# ── 5.2  Parse date columns ───────────────────────────────────────────────────
for col in ['created_at', 'updated_at', 'processed_at', 'scheduled_at', 'shipped_date']:
    if col in df.columns:
        df[col] = safe_to_datetime(df[col])

# ── 5.3  Cast numeric columns ─────────────────────────────────────────────────
for col in ['total_price', 'subtotal_price', 'total_tax', 'total_discounts',
            'total_refunds', 'total_weight']:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

# ── 5.4  Normalise status ─────────────────────────────────────────────────────
for col in ['status', 'type', 'payment_processor']:
    if col in df.columns:
        df[col] = df[col].str.strip().str.lower()

silver_rc_orders = df
cleaning_report(df_raw, silver_rc_orders, 'silver_rc_orders')

silver_rc_orders.to_parquet(os.path.join(SILVER_DIR, 'silver_rc_orders.parquet'), index=False)
print("✅ Saved: silver_rc_orders.parquet")

---
## 6. Recharge Checkout Items → `silver_rc_items_checkout`

In [ ]:
df = pd.read_parquet(os.path.join(BRONZE_DIR, 'bronze_rc_items_checkout.parquet'))
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")

# Deduplicate on composite key
key_cols = ['recharge_order_id', 'product_sku']
if all(c in df.columns for c in key_cols):
    df = df.drop_duplicates(subset=key_cols, keep='first')

# Normalise SKU
if 'product_sku' in df.columns:
    df['product_sku'] = df['product_sku'].str.strip().str.upper()

# Cast numerics
for col in ['quantity', 'unit_price', 'total_price']:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

silver_rc_items_checkout = df
cleaning_report(df_raw, silver_rc_items_checkout, 'silver_rc_items_checkout')

silver_rc_items_checkout.to_parquet(os.path.join(SILVER_DIR, 'silver_rc_items_checkout.parquet'), index=False)
print("✅ Saved: silver_rc_items_checkout.parquet")

---
## 7. Recharge Recurring Items → `silver_rc_items_recurring`

In [ ]:
df = pd.read_parquet(os.path.join(BRONZE_DIR, 'bronze_rc_items_recurring.parquet'))
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")

key_cols = ['recharge_order_id', 'product_sku']
if all(c in df.columns for c in key_cols):
    df = df.drop_duplicates(subset=key_cols, keep='first')

if 'product_sku' in df.columns:
    df['product_sku'] = df['product_sku'].str.strip().str.upper()

for col in ['quantity', 'unit_price', 'total_price']:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

silver_rc_items_recurring = df
cleaning_report(df_raw, silver_rc_items_recurring, 'silver_rc_items_recurring')

silver_rc_items_recurring.to_parquet(os.path.join(SILVER_DIR, 'silver_rc_items_recurring.parquet'), index=False)
print("✅ Saved: silver_rc_items_recurring.parquet")

---
## 8. Recharge Reactivated Subs → `silver_rc_reactivated`

In [ ]:
df = pd.read_parquet(os.path.join(BRONZE_DIR, 'bronze_rc_reactivated.parquet'))
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")

if 'customer_id' in df.columns:
    df = df.drop_duplicates(subset=['customer_id'], keep='first')

for col in ['reactivated_at', 'created_at', 'updated_at', 'cancelled_at', 'next_charge_scheduled_at']:
    if col in df.columns:
        df[col] = safe_to_datetime(df[col])

for col in ['price', 'quantity']:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

silver_rc_reactivated = df
cleaning_report(df_raw, silver_rc_reactivated, 'silver_rc_reactivated')

silver_rc_reactivated.to_parquet(os.path.join(SILVER_DIR, 'silver_rc_reactivated.parquet'), index=False)
print("✅ Saved: silver_rc_reactivated.parquet")

---
## 9. Recharge Churned Subs → `silver_rc_churned`

In [ ]:
df = pd.read_parquet(os.path.join(BRONZE_DIR, 'bronze_rc_churned.parquet'))
df_raw = df.copy()
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")

if 'subscription_id' in df.columns:
    df = df.drop_duplicates(subset=['subscription_id'], keep='first')

for col in ['created_at', 'updated_at', 'cancelled_at', 'next_charge_scheduled_at']:
    if col in df.columns:
        df[col] = safe_to_datetime(df[col])

for col in ['price', 'quantity']:
    if col in df.columns:
        df[col] = safe_to_numeric(df[col])

if 'cancellation_reason' in df.columns:
    df['cancellation_reason'] = df['cancellation_reason'].str.strip().str.lower()

silver_rc_churned = df
cleaning_report(df_raw, silver_rc_churned, 'silver_rc_churned')

silver_rc_churned.to_parquet(os.path.join(SILVER_DIR, 'silver_rc_churned.parquet'), index=False)
print("✅ Saved: silver_rc_churned.parquet")

---
## 10. Cross-table Validation

In [ ]:
# ── Check: all SKUs in order line items exist in product master ───────────────
if 'Line: SKU' in silver_orders.columns and 'Variant SKU' in silver_products.columns:
    order_skus   = set(silver_orders['Line: SKU'].dropna().str.strip().str.upper())
    product_skus = set(silver_products['Variant SKU'].dropna())
    unmatched    = order_skus - product_skus
    print(f"SKUs in orders not found in product master: {len(unmatched):,}")
    if unmatched:
        print("  Sample:", list(unmatched)[:10])

# ── Check: Recharge order IDs in items exist in rc_orders ─────────────────────
if 'recharge_order_id' in silver_rc_orders.columns and 'recharge_order_id' in silver_rc_items_checkout.columns:
    rc_order_ids    = set(silver_rc_orders['recharge_order_id'].dropna())
    rc_item_ids     = set(silver_rc_items_checkout['recharge_order_id'].dropna())
    orphan_items    = rc_item_ids - rc_order_ids
    print(f"Checkout items with no matching RC order: {len(orphan_items):,}")

---
## 11. Silver Layer Summary

In [ ]:
silver_tables = {
    'silver_orders'              : silver_orders,
    'silver_products'            : silver_products,
    'silver_discounts'           : silver_discounts,
    'silver_sessions'            : silver_sessions,
    'silver_rc_orders'           : silver_rc_orders,
    'silver_rc_items_checkout'   : silver_rc_items_checkout,
    'silver_rc_items_recurring'  : silver_rc_items_recurring,
    'silver_rc_reactivated'      : silver_rc_reactivated,
    'silver_rc_churned'          : silver_rc_churned,
}

print(f"{'Table':<35} {'Rows':>8}  {'Cols':>6}")
print('─' * 55)
for name, df in silver_tables.items():
    print(f"{name:<35} {df.shape[0]:>8,}  {df.shape[1]:>6}")

print(f"\n✅ All silver tables saved to: {SILVER_DIR}")